# Embedding Fine-Tuning — run on Kaggle

**Before running:** open the right sidebar → **Settings** and set:
- **Accelerator = GPU T4 ×2** (or P100)
- **Internet = On** (needed to pip-install and download FiQA + the base model)
- *(optional)* Add-ons → **Secrets** → add `OPENAI_API_KEY` if you want the LLM triplet path.

There are two paths below:
- **Path A (recommended):** run the whole pipeline headless, read the before/after numbers.
- **Path B (optional):** launch the FastAPI + Streamlit UI behind a public `cloudflared` URL.

## 1. Get the code + install dependencies

In [ ]:
# Public repo - clones on Kaggle with no auth. (Or upload as a Kaggle Dataset and
# `%cd /kaggle/input/<dataset-name>` instead of cloning.)
!git clone https://github.com/Om-merkle/AI_EMBEDDING_FINETUNE.git
%cd AI_EMBEDDING_FINETUNE
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Path A — full pipeline (headless)
Runs: prepare data → mine triplets → **MTEB baseline** → **fine-tune** → evaluate → compare.

In [ ]:
!python run_pipeline.py --domain fiqa --base-model BAAI/bge-small-en-v1.5 --epochs 1 --batch-size 32

In [ ]:
import json
print(json.dumps(json.load(open('results/comparison.json')), indent=2))

The fine-tuned model is saved under `models/` and metrics under `results/` — both appear
in the notebook's **Output** tab for download. (Optional: `model.push_to_hub(...)` with an
`HF_TOKEN` secret.)

## 3. Path B (optional) — launch the UI behind a public URL
Starts the FastAPI backend + Streamlit UI in the background and opens a `cloudflared`
tunnel. Watch the output for a `https://<random>.trycloudflare.com` link.

In [ ]:
import subprocess, time, re, os, threading

# 1) backend + frontend in the background
subprocess.Popen(['uvicorn', 'api.main:app', '--port', '8000'])
subprocess.Popen(['streamlit', 'run', 'app/streamlit_app.py',
                  '--server.port', '8501', '--server.headless', 'true'],
                 env={**os.environ, 'API_URL': 'http://localhost:8000'})
time.sleep(15)

# 2) cloudflared tunnel to the Streamlit port
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n>>> OPEN THE UI:', m.group(0))
        break